# Build the priors

Turns each downloaded prior into one of the two forms that [`pwpa.py`](pwpa.py) reads. PIE's five sources are all profiles.

- **profile**, `priors/profile/<source>.parquet`: one row per gene. The similarity of two genes is the dot product of their rows, so the rows are scaled first. A pair is covered when both genes have a row.
- **network**, `priors/network/<source>.parquet`: weighted gene pairs. A missing pair has similarity 0 and still counts as covered.

At fit time `pwpa.py` z-scores every prior over the training pairs, which puts all priors on a common scale.

For PIE's sources the similarity is the Pearson correlation of the two genes' DepMap gene effects, and the cosine of the two embeddings for the other four.
`ncbi_text` stores one vector per token, and a gene's vector is their mean.
Rows are indexed by human gene symbol, the key PIE looks genes up by, so the same files serve every dataset; the mouse datasets name each knockdown by its human orthologue.
As in PIE, a gene missing from a source is looked up again through the source's `aliases.yaml`.

In [ ]:
import json
import sys

import numpy as np
import pandas as pd

sys.path.append("../utils")
from paths import data_dir

import pwpa

In [ ]:
def standard_rows(table):
    """Profile rows from a numeric gene x feature table (screens, DepMap, expression): each row centred and scaled to
    unit length, so the dot product of two rows is their Pearson correlation. A missing value counts as the row mean."""
    return unit_rows(table.sub(table.mean(axis=1), axis=0).fillna(0))


def unit_rows(X):
    """Rows scaled to unit length, so the dot product of two rows is their cosine. A row of zeros carries no
    information, so its gene is dropped (left uncovered)."""
    norm = np.linalg.norm(X, axis=1)
    return X[norm > 0].div(norm[norm > 0], axis=0)

In [ ]:
raw = data_dir("priors", "raw")
for source in ["perturbation_text", "ncbi_text", "esm2", "string_space", "depmap_gene_effect"]:
    meta = json.loads((raw / source / "meta.json").read_text())
    emb = np.load(raw / source / "embeddings.npy", mmap_mode="r")
    if meta["layout"] == "token":  # ncbi_text, 43 GB: read one gene's tokens at a time
        offsets = np.load(raw / source / "offsets.npy")
        vectors = np.stack([emb[a:b].mean(0, dtype=np.float64) for a, b in zip(offsets[:-1], offsets[1:])])
    else:
        vectors = np.asarray(emb, np.float64)
    table = pd.DataFrame(vectors, index=pd.Index(meta["keys"], name="gene")).rename(columns=str)

    # PIE uses an alias only when the name itself is missing and the alias target is present
    path = raw / source / "aliases.yaml"
    aliases = dict(line.split(": ") for line in path.read_text().splitlines()) if path.exists() else {}  # flat "a: b"
    extra = {a: t for a, t in aliases.items() if a not in table.index and t in table.index}
    table = pd.concat([table, table.loc[list(extra.values())].set_axis(pd.Index(list(extra), name="gene"))])

    rows = standard_rows(table) if source == "depmap_gene_effect" else unit_rows(table)
    assert np.allclose(np.linalg.norm(rows, axis=1), 1, atol=1e-5), source
    rows.astype(np.float32).to_parquet(data_dir("priors", "profile") / f"{source}.parquet")
    print(source, rows.shape)

In [ ]:
priors = pwpa.load_priors(data_dir("priors"))
print(len(priors), "priors:", ", ".join(priors))